# FaithfulRecon — Colab runner

Runs the experiments on a Colab GPU: base models + signals, D2 protocol v2.1, analysis, D1 run dirs,
for **DTU** and **NeRF-Synthetic**.

Before running:
1. Runtime → Change runtime type → GPU.
2. Colab **Secrets** (key icon on the left): add `GH_TOKEN` = a GitHub fine-grained token with
   *read-only* access to the repo (Contents: read). Do not paste the token into this notebook.
3. Fill the dataset URLs in the *Data* section (official sources; links are not hard-coded on purpose).

Everything heavy (`runs/`, `data/`) lives on Google Drive, so a disconnected session resumes where it
stopped: just re-run the cells. All experiment stages are cached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Setup

In [ ]:
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')

STORE = '/content/drive/MyDrive/faithfulrecon_store'   # persistent: runs/, data/, results copies
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'main'                                        # or a feature branch
REPO = '/content/faithfulrecon'

os.makedirs(f'{STORE}/runs', exist_ok=True)
os.makedirs(f'{STORE}/data', exist_ok=True)
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
# keep the token out of the saved remote URL
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
for d in ('runs', 'data'):
    p = pathlib.Path(REPO, d)
    if p.is_symlink() or not p.exists():
        if p.is_symlink(): p.unlink()
        p.symlink_to(f'{STORE}/{d}')
    elif p.is_dir() and not any(x for x in p.iterdir() if x.name != 'README.md'):
        import shutil; shutil.rmtree(p); p.symlink_to(f'{STORE}/{d}')
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip -q install -r requirements.txt trimesh rtree
!python -m pytest -q

### Optional: gsplat backend (faster)
Only if the parity test passes; then set `renderer: gsplat` in the configs. On recent GPUs gsplat may
compile its CUDA kernels on first use (several minutes).

In [ ]:
USE_GSPLAT = False
if USE_GSPLAT:
    !pip -q install gsplat
    !python -m pytest -q -s tests/test_gsplat_parity.py

## Data

### DTU
- **Scans** (images, masks, `cameras_sphere.npz` / `cameras.npz`): the IDR / NeuS / 2DGS preprocessed
  DTU release — take the link from the README of the repo you use (e.g. 2DGS or NeuS).
- **Reference geometry**: official DTU MVS 2014 *Points* archive (STL point clouds),
  https://roboimagedata.compute.dtu.dk/?page_id=36 (≈ 6.3 GB). Only the STL files of the chosen scans
  are extracted.

Expected layout after this cell: `data/dtu/scan24/{image,mask,cameras*.npz}` and
`data/dtu/Points/stl/stl024_total.ply`.

In [ ]:
DTU_SCANS = [24]                 # add more scans later, e.g. [24, 37, 40, 55, 63, 65]
DTU_PREPROC_URL = ''             # zip of the preprocessed DTU scans
DTU_POINTS_URL = ''              # official Points.zip from the DTU page

import glob, zipfile
os.makedirs('data/dtu', exist_ok=True)
if DTU_PREPROC_URL and not all(os.path.exists(f'data/dtu/scan{s}/image') for s in DTU_SCANS):
    !wget -q -O /content/dtu_preproc.zip "{DTU_PREPROC_URL}"
    !unzip -q -o /content/dtu_preproc.zip -d /content/dtu_preproc
    # locate each scan (folder names differ between releases: scan24, dtu_scan24, ...)
    for s in DTU_SCANS:
        hits = [os.path.dirname(p) for p in glob.glob('/content/dtu_preproc/**/cameras*.npz', recursive=True)
                if os.path.basename(os.path.dirname(p)).endswith(f'scan{s}')]
        assert hits, f'scan{s} not found in the archive'
        !mkdir -p data/dtu/scan{s} && cp -r {hits[0]}/* data/dtu/scan{s}/
if DTU_POINTS_URL and not all(os.path.exists(f'data/dtu/Points/stl/stl{s:03d}_total.ply') for s in DTU_SCANS):
    !wget -q -O /content/dtu_points.zip "{DTU_POINTS_URL}"
    with zipfile.ZipFile('/content/dtu_points.zip') as z:
        for s in DTU_SCANS:
            name = next(n for n in z.namelist() if n.endswith(f'stl{s:03d}_total.ply'))
            os.makedirs('data/dtu/Points/stl', exist_ok=True)
            with z.open(name) as src, open(f'data/dtu/Points/stl/stl{s:03d}_total.ply', 'wb') as dst:
                dst.write(src.read())
!ls data/dtu data/dtu/Points/stl 2>/dev/null

### NeRF-Synthetic
- **Images + cameras**: official NeRF release (Google Drive, `nerf_synthetic.zip`).
- **Reference geometry is not part of the dataset.** If a mesh in the Blender world frame is
  available (e.g. exported from a .blend with `scripts/blender_export_mesh.py`), put it at
  `data/reference/lego.ply`; the cell samples visible-surface points by ray casting and the config
  must then set `scene.ref_points_path`. Without it, D2 runs without the geometry terms.

In [ ]:
NERF_SYNTH_GDRIVE = ''           # share link of nerf_synthetic.zip
if NERF_SYNTH_GDRIVE and not os.path.exists('data/nerf_synthetic/lego'):
    !pip -q install gdown
    !gdown --fuzzy "{NERF_SYNTH_GDRIVE}" -O /content/nerf_synthetic.zip
    !unzip -q -o /content/nerf_synthetic.zip -d data/
if os.path.exists('data/reference/lego.ply') and not os.path.exists('data/reference/lego_points.npy'):
    !python scripts/make_reference_points.py --config configs/blender_lego_v8.yaml \
        --mesh data/reference/lego.ply --out data/reference/lego_points.npy --stride 2
!ls data/nerf_synthetic 2>/dev/null | head

## Run
`WORKERS` processes share the GPU (each run uses a few GB). Start with 6 and watch `nvidia-smi`;
increase while utilization is below ~90 % and memory allows.

In [ ]:
WORKERS = 6

def pipeline(base_cfg, d2_cfg, scene, condition, seeds='0 1 2 3 4 5'):
    run_dir = d2_cfg.replace('configs/', 'runs/').replace('.yaml', '')
    !python scripts/run_parallel.py --script scripts/phase0.py --config {base_cfg} --workers {WORKERS} --base_only
    !python scripts/run_parallel.py --script scripts/d2_v2.py --config {d2_cfg} --workers {WORKERS}
    !python -W ignore scripts/analyze_d2.py --run_dir {run_dir} > {run_dir}/analyze.txt; tail -n 12 {run_dir}/analyze.txt
    base_dir = base_cfg.replace('configs/', 'runs/').replace('.yaml', '')
    !python scripts/prepare_d1_run.py --phase0_dir {base_dir} --seeds {seeds} --scene {scene} --condition {condition}

In [ ]:
# DTU scan24, 8 views
pipeline('configs/dtu_scan24_v8.yaml', 'configs/d2v21_dtu_scan24_v8.yaml', 'dtu_scan24', 'v8')

In [ ]:
# NeRF-Synthetic lego, 8 views (geometry terms only if ref_points_path is set in the config)
pipeline('configs/blender_lego_v8.yaml', 'configs/d2v21_blender_lego_v8.yaml', 'blender_lego', 'v8')

## Results

In [ ]:
from IPython.display import Image, display
for d in ['runs/d2v21_dtu_scan24_v8', 'runs/d2v21_blender_lego_v8']:
    f = f'{d}/d2_figure.png'
    if os.path.exists(f):
        print(d); display(Image(f, width=1100))
# small results (report + figure) also go to Drive/results for sharing
!mkdir -p {STORE}/results && for d in runs/d2v21_*; do mkdir -p {STORE}/results/$(basename $d) && cp $d/report.md $d/d2_figure.png {STORE}/results/$(basename $d)/ 2>/dev/null; done
!ls {STORE}/results